# Microduck: policy standard su pavimento piano

Esegue `alpha_walking.onnx` (la policy di camminata ufficiale) nell'ambiente `Mjlab-Velocity-Flat-MicroDuck` di microduck_rl e salva un video della camminata.

**Prima di iniziare:** *Runtime → Change runtime type → T4 GPU*. Poi esegui le celle in ordine.

## 1. Controllo della GPU

In [ ]:
!nvidia-smi

## 2. Installazione

Installa `uv`, clona microduck_rl e installa le dipendenze (versioni fissate in `uv.lock`). Il primo `uv sync` scarica qualche GB e richiede alcuni minuti.

In [ ]:
import os

!curl -LsSf https://astral.sh/uv/install.sh | sh
os.environ["PATH"] = "/root/.local/bin:" + os.environ["PATH"]
os.environ["UV_HTTP_TIMEOUT"] = "600"   # evita timeout sui pacchetti CUDA grandi

if not os.path.exists("/content/microduck_rl"):
    !git clone https://github.com/pollen-robotics/microduck_rl /content/microduck_rl
%cd /content/microduck_rl
!uv sync

In [ ]:
# Verifica che torch nell'ambiente del progetto veda la GPU (deve stampare True)
!uv run python -c "import torch; print('CUDA:', torch.cuda.is_available(), torch.version.cuda)"

## 3. Rendering senza schermo (EGL)

Colab non ha un display: MuJoCo renderizza i frame sulla GPU tramite EGL. La cella registra il driver NVIDIA per EGL se manca (stesso trucco dei notebook ufficiali di MuJoCo).

In [ ]:
NVIDIA_ICD_CONFIG_PATH = "/usr/share/glvnd/egl_vendor.d/10_nvidia.json"
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    os.makedirs(os.path.dirname(NVIDIA_ICD_CONFIG_PATH), exist_ok=True)
    with open(NVIDIA_ICD_CONFIG_PATH, "w") as f:
        f.write('{"file_format_version": "1.0.0", "ICD": {"library_path": "libEGL_nvidia.so.0"}}\n')

os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = "egl"

## 4. Scaricare la policy standard

`alpha_walking.onnx` sta nel repo Hugging Face `pollen-robotics/microduck-policies`. Prima si elencano i file per trovare il percorso esatto. Se ricevi un errore 401/404 il repo è privato: esegui `!uv run huggingface-cli login` con un token che ha accesso, oppure carica il file a mano nella cartella `/content/microduck_rl`.

In [ ]:
!uv run python -c "from huggingface_hub import list_repo_files; print('\n'.join(list_repo_files('pollen-robotics/microduck-policies')))"

In [ ]:
# Se nella lista il file è in una sottocartella, scrivi qui il percorso completo
POLICY_FILE = "alpha_walking.onnx"

!uv run python -c "from huggingface_hub import hf_hub_download; print(hf_hub_download('pollen-robotics/microduck-policies', '{POLICY_FILE}', local_dir='policies'))"
POLICY_PATH = os.path.join("policies", POLICY_FILE)
print("policy:", POLICY_PATH, os.path.exists(POLICY_PATH))

## 5. Lo script di simulazione

- l'ambiente è la configurazione della camminata su piano (`make_microduck_velocity_env_cfg(play=True, rough=False)`);
- la policy ONNX viene eseguita come rete PyTorch con `OnnxMLP` di `scripts/pickup_datagen.py` (normalizzazione delle osservazioni inclusa);
- le spinte vengono tolte: in modalità play arrivano ogni 0,5-1 s, utili per testare ma non per guardare la camminata;
- i comandi di velocità sono casuali e ricampionati durante l'episodio.

In [ ]:
%%writefile cammina_piano.py
import argparse, sys

import imageio
import numpy as np
import torch

sys.path.insert(0, "scripts")
import mjlab_microduck.tasks  # noqa: F401  registra i task
from mjlab.envs import ManagerBasedRlEnv
from pickup_datagen import OnnxMLP
from mjlab_microduck.tasks.microduck_velocity_env_cfg import make_microduck_velocity_env_cfg

p = argparse.ArgumentParser()
p.add_argument("policy")
p.add_argument("--seconds", type=float, default=10.0)
p.add_argument("--pushes", action="store_true", help="lascia le spinte della modalità play")
p.add_argument("--out", default="cammina_piano.mp4")
args = p.parse_args()

dev = "cuda:0" if torch.cuda.is_available() else "cpu"
print("device:", dev)

cfg = make_microduck_velocity_env_cfg(play=True, rough=False)
cfg.scene.num_envs = 1
if not args.pushes:
    cfg.events.pop("push_robot", None)

env = ManagerBasedRlEnv(cfg=cfg, device=dev, render_mode="rgb_array")
policy = OnnxMLP(args.policy, dev)


def frame():
    f = env.render()
    if hasattr(f, "cpu"):
        f = f.cpu().numpy()
    f = np.asarray(f)
    return f[0] if f.ndim == 4 else f  # una immagine per copia -> la prima


obs, _ = env.reset()
assert obs["actor"].shape[-1] == 61, f"osservazioni {obs['actor'].shape}: la policy si aspetta 61"
frames, cadute = [], 0
for t in range(int(args.seconds / env.step_dt)):
    obs, _, term, trunc, _ = env.step(policy(obs["actor"]))
    frames.append(frame())
    if t % 50 == 0:
        vx, vy, wz = obs["actor"][0, 48:51].tolist()
        print(f"t={t * env.step_dt:5.1f}s  comando vx={vx:+.2f} vy={vy:+.2f} wz={wz:+.2f}")
    if term.any():
        cadute += 1
        print(f"caduto a t={t * env.step_dt:.1f}s")

imageio.mimsave(args.out, frames, fps=round(1 / env.step_dt))
print(f"cadute: {cadute} | video: {args.out} ({len(frames)} frame)")
env.close()

## 6. Esecuzione

La prima esecuzione compila i kernel di MuJoCo Warp (qualche minuto); le successive sono veloci finché il runtime non si riavvia.

In [ ]:
!uv run --with "imageio[ffmpeg]" python cammina_piano.py {POLICY_PATH} --seconds 10

## 7. Video

In [ ]:
from IPython.display import Video
Video("cammina_piano.mp4", embed=True, width=640)

## Se qualcosa non va

| Errore | Causa probabile | Cosa fare |
|---|---|---|
| `CUDA: False` nella cella 2 | Runtime senza GPU | *Runtime → Change runtime type → T4 GPU*, poi rieseguire tutto |
| Errore EGL / `gladLoadGL` in `env.render()` | Rendering EGL non disponibile | Rieseguire la cella 3; in alternativa `!apt-get install -y libosmesa6` e `MUJOCO_GL=osmesa` |
| `ValueError` con un nome di operazione da `OnnxMLP` | Il grafo ONNX usa operazioni diverse da Sub/Div/Gemm/Elu | Estendere `OnnxMLP` (sezione 4.5 del piano) |
| Assert sulle 61 osservazioni | Policy vecchia a 51 osservazioni | Usare una policy con il layout a 61 |
| Il robot cade subito | Policy o osservazioni non allineate | Controllare di usare la policy di camminata e non quella di stand |